In [2]:
# Colab only: clone the repo and move into the project root
import os
import sys

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    # Clone repository if it hasn't been cloned yet
    if not os.path.exists('/content/convolutional_vae_quishing'):
        !git clone https://github.com/AnhJoe/convolutional_vae_quishing.git

    # Change working directory to the project root
    os.chdir('/content/convolutional_vae_quishing')

In [3]:
#| output: false
# Local only: find the project root (the folder containing src/) and add it to the import path
# For local runs. Else if running Colab, skip this.

import sys
from pathlib import Path

# Set to root for src load
root = Path.cwd()
while root != root.parent and not (root / "src").exists():
    root = root.parent

if not (root / "src").exists():
    raise FileNotFoundError("Could not find 'src' folder above current working directory.")

sys.path.insert(0, str(root))
print("Project root:", root)

Project root: c:\Users\joetn\convolutional_vae_quishing


# Data processing

This notebook turns the cleaned manifest from `1_eda.ipynb` into one shared table, the **items table**, that every model notebook reads. Nothing here trains a model. It is run once, and the model notebooks (`3_image_vae.ipynb` and `4_url_vae.ipynb`) only read its output, so everyone trains, tunes and tests on identical data.

Run `1_eda.ipynb` first. It downloads the dataset into `data/raw` and saves `data/processed/manifest.csv.gz`, the list of every QR code with its cleaning decision. This notebook then does four things:

1. Counts which QR codes fit the 65 x 65 image canvas decided in `1_eda.ipynb` section 4.
2. Measures every QR image (size, quiet zone, dark fraction), which the image strata and the image floor need.
3. Assigns every code to train, validation or test, using the grouped rule decided in `1_eda.ipynb` section 6, and saves the items table.
4. Checks the splits.

Steps 1 and 2 read about a million image files, so they are slow (step 1 reads only the file header and is much faster than step 2). Both can be stopped and started again, and both skip work that is already saved. Everything is saved in `data/processed/`, which is git-ignored. The column-by-column description of the items table is at the top of `src/prepare_data.py`.

## 1. Load the cleaned manifest

The manifest has one row per URL from the source CSVs. Its `keep` column is false only for the 38 rows whose URL appears in both classes (`1_eda.ipynb` section 7), so the items table has 1,005,700 rows.

In [4]:
# Imports, paths and the time limit for the slow steps
import json

import numpy as np
import pandas as pd

from src import canvas_exclusion_count, prepare_data
from src.splits import SPLIT_NAMES

PROC_DIR = root / "data" / "processed"
RAW_DIR = root / "data" / "raw"

# None means no limit. In a time-limited session (for example Colab) set a number of seconds, such as 1200.
# The slow steps then stop politely after that time, and running the same cell again continues where it stopped.
MAX_SECONDS = None

if not (PROC_DIR / "manifest.csv.gz").exists():
    raise FileNotFoundError("data/processed/manifest.csv.gz not found. Run 1_eda.ipynb first.")
manifest = pd.read_csv(PROC_DIR / "manifest.csv.gz", usecols=["cls", "keep", "drop_reason"], keep_default_na=False)
print(f"{len(manifest):,} rows in the manifest, {int(manifest['keep'].sum()):,} kept")
manifest.groupby("cls")["keep"].agg(rows="size", kept="sum")

1,005,738 rows in the manifest, 1,005,700 kept


,rows,kept
cls,,
benign,429976,429957
malicious,575762,575743


## 2. Which codes fit the image canvas

The image VAE takes one fixed input size, a 65 x 65 grid of modules (a module is one black or white square of a QR code). A code whose grid, quiet zone included, is larger than that cannot be used without cropping, so it is excluded from the image model and counted. URL models use every row.

`canvas_exclusion_count.run` reads the size of every kept PNG (the 1,005,700 rows of the items table). It reads only the first bytes of each file (the header, not the pixels), and every image is drawn at 6 pixels per module, so the grid size is the image width divided by 6. It saves `image_grid_sizes.csv.gz` (one row per image) and `canvas_exclusion_summary.json` (counts per class).

In [5]:
# Measure the grid size of every kept image (header only). Skipped if the result is already saved
grid_file = PROC_DIR / "image_grid_sizes.csv.gz"
if grid_file.exists():
    print("already measured:", grid_file.name, "(delete the file to measure again)")
else:
    canvas_exclusion_count.run(root, max_seconds=MAX_SECONDS)

1,005,700 images, 0 already measured, 1,005,700 to do
  measured 5,000 of 1,005,700 in 1 s
  measured 10,000 of 1,005,700 in 2 s
  measured 15,000 of 1,005,700 in 3 s
  measured 20,000 of 1,005,700 in 4 s
  measured 25,000 of 1,005,700 in 5 s
  measured 30,000 of 1,005,700 in 6 s
  measured 35,000 of 1,005,700 in 7 s
  measured 40,000 of 1,005,700 in 8 s
  measured 45,000 of 1,005,700 in 9 s
  measured 50,000 of 1,005,700 in 10 s
  measured 55,000 of 1,005,700 in 12 s
  measured 60,000 of 1,005,700 in 13 s
  measured 65,000 of 1,005,700 in 13 s
  measured 70,000 of 1,005,700 in 14 s
  measured 75,000 of 1,005,700 in 16 s
  measured 80,000 of 1,005,700 in 16 s
  measured 85,000 of 1,005,700 in 18 s
  measured 90,000 of 1,005,700 in 18 s
  measured 95,000 of 1,005,700 in 20 s
  measured 100,000 of 1,005,700 in 20 s
  measured 105,000 of 1,005,700 in 21 s
  measured 110,000 of 1,005,700 in 22 s
  measured 115,000 of 1,005,700 in 23 s
  measured 120,000 of 1,005,700 in 25 s
  measured 125,

In [6]:
# Codes outside the 65 x 65 canvas, per class, over all kept images
cs = json.loads((PROC_DIR / "canvas_exclusion_summary.json").read_text())
excluded = pd.DataFrame({
    cls: {"images": s["images"], "outside the canvas": s["outside_canvas"], "outside share": round(s["outside_canvas_share"], 4),
          "largest grid (modules)": s["largest_grid"], "unreadable or irregular files": sum(s["irregular_or_unreadable"].values())}
    for cls, s in cs["per_class"].items()})
display(excluded)

,benign,malicious
images,429957.0000,575743.0000
outside the canvas,157.0000,4506.0000
outside share,0.0004,0.0078
largest grid (modules),125.0000,169.0000
unreadable or irregular files,0.0000,0.0000


**Reading.** The first table is the count of excluded codes over all kept images.

## 3. Measure every QR image

The image strata and the image floor need three properties of every code: its number of modules per side (which gives the QR version, with Micro QR codes below 21 modules as their own group), the width of its quiet zone, and the fraction of dark pixels. `prepare_data.run_images` measures them with the method used in `1_eda.ipynb` section 4, for every kept image, and saves `image_features.csv.gz`. This reads every PNG, so it is the slowest step. Run it again to continue after a stop.

In [7]:
# Measure modules, quiet zone and dark fraction of every kept image. Skipped if the result is already saved
features_file = PROC_DIR / prepare_data.IMAGE_FEATURES_FILE
if features_file.exists():
    print("already measured:", features_file.name, "(delete the file to measure again)")
else:
    prepare_data.run_images(root, max_seconds=MAX_SECONDS)

1,005,700 images, 0 already measured, 1,005,700 to do
  measured 2,000 of 1,005,700 in 2 s
  measured 4,000 of 1,005,700 in 4 s
  measured 6,000 of 1,005,700 in 5 s
  measured 8,000 of 1,005,700 in 7 s
  measured 10,000 of 1,005,700 in 8 s
  measured 12,000 of 1,005,700 in 9 s
  measured 14,000 of 1,005,700 in 11 s
  measured 16,000 of 1,005,700 in 12 s
  measured 18,000 of 1,005,700 in 14 s
  measured 20,000 of 1,005,700 in 15 s
  measured 22,000 of 1,005,700 in 17 s
  measured 24,000 of 1,005,700 in 19 s
  measured 26,000 of 1,005,700 in 20 s
  measured 28,000 of 1,005,700 in 22 s
  measured 30,000 of 1,005,700 in 23 s
  measured 32,000 of 1,005,700 in 25 s
  measured 34,000 of 1,005,700 in 26 s
  measured 36,000 of 1,005,700 in 28 s
  measured 38,000 of 1,005,700 in 29 s
  measured 40,000 of 1,005,700 in 30 s
  measured 42,000 of 1,005,700 in 32 s
  measured 44,000 of 1,005,700 in 33 s
  measured 46,000 of 1,005,700 in 34 s
  measured 48,000 of 1,005,700 in 35 s
  measured 50,000 of

In [8]:
# Summary of the image measurements per class
cols = ["side", "modules", "border", "dark_frac"]
feat = pd.read_csv(features_file, keep_default_na=False, na_values={c: [""] for c in cols})
bad = feat["error"].fillna("").ne("")
print(f"{len(feat):,} images measured, {int(bad.sum()):,} could not be measured")
ok = feat[~bad]
pd.DataFrame({
    "images": ok.groupby("cls").size(),
    "Micro QR share (under 21 modules)": ok.groupby("cls")["modules"].apply(lambda s: (s < 21).mean()).round(4),
    "median modules per side": ok.groupby("cls")["modules"].median(),
    "largest modules per side": ok.groupby("cls")["modules"].max(),
    "median dark fraction": ok.groupby("cls")["dark_frac"].median().round(4),
}).T

1,005,700 images measured, 0 could not be measured


cls,benign,malicious
images,429957.0000,575743.0000
Micro QR share (under 21 modules),0.0243,0.0675
median modules per side,29.0000,29.0000
largest modules per side,117.0000,161.0000
median dark fraction,0.3156,0.3097


## 4. Build the items table

One row per kept QR code. The table joins the manifest (URL, label and URL features), the split, the canvas fit and the image measurements, and saves them to `items.csv.gz`.

**The split rule** was decided in `1_eda.ipynb` section 6 and is applied here without change:

- Benign and malicious rows are each split 70/15/15 into train, validation and test (70% for training, 15% for tuning, 15% for the final test).
- The split is grouped by domain (the site name, for example `wikipedia.org`), so all rows of one domain stay in one split. This stops a model from scoring well just because it saw other pages of the same site during training.
- The largest domains fill train first, and the remaining domains are shuffled and dealt to validation and test, with seed 0.

`prepare_data.run_items` also checks that every domain sits in one split, that item ids run from 0 to n-1, and that labels match the class folder.

In [9]:
# Build and save the items table and its summary
prepare_data.run_items(root)

saved c:\Users\joetn\convolutional_vae_quishing\data\processed\items.csv.gz: 1,005,700 items, data version 0781657d11c5, image features filled for 1,005,700 items
saved c:\Users\joetn\convolutional_vae_quishing\data\processed\items_summary.json


In [10]:
# Read the saved table back the way the model notebooks do, and look at the first rows
items = prepare_data.load_items(PROC_DIR)
print(items.shape)
items.head()

(1005700, 26)


,item_id,cls,label,index,file,url,split,domain,host,fits_canvas,...,is_https,host_is_ip,n_host_labels,path_len,digit_ratio,side,modules,border,dark_frac,qr_version
0,0,benign,0,1,benign_000001.png,http://freshrpms.net/,train,freshrpms.net,freshrpms.net,True,...,False,False,2,1,0.000000,198,25,4.0,0.303030,2.0
1,1,benign,0,2,benign_000002.png,http://lists.freshrpms.net/mailman/listinfo/rp...,train,freshrpms.net,lists.freshrpms.net,True,...,False,False,3,26,0.000000,222,29,4.0,0.316289,3.0
2,2,benign,0,3,benign_000003.png,http://www.boquist.net/stort-sup-brev,validation,boquist.net,www.boquist.net,True,...,False,False,3,15,0.000000,222,29,4.0,0.314098,3.0
3,3,benign,0,6,benign_000006.png,http://hesketh.com,test,hesketh.com,hesketh.com,True,...,False,False,2,0,0.000000,198,25,4.0,0.304867,2.0
4,4,benign,0,7,benign_000007.png,http://seeker.dice.com/seeker.epl?rel_code=31,validation,dice.com,seeker.dice.com,True,...,False,False,3,11,0.044444,222,29,4.0,0.315559,3.0


## 5. Checks on the splits

Two checks that must hold: no domain appears in more than one split, and no validation or test row shares a host with a train row of its own class. Then the counts per split.

In [11]:
# Checks (each domain sits in one split, no host is shared with train)
checks = {}
for cls, g in items.groupby("cls"):
    train_hosts = set(g.loc[g["split"] == "train", "host"])
    checks[cls] = {
        "domains found in more than one split": int((g.groupby("domain")["split"].nunique() > 1).sum()),
        "validation or test rows sharing a host with train": int(g.loc[g["split"] != "train", "host"].isin(train_hosts).sum()),
    }
display(pd.DataFrame(checks))
assert all(v == 0 for c in checks.values() for v in c.values())

,benign,malicious
domains found in more than one split,0,0
validation or test rows sharing a host with train,0,0


In [12]:
# Class counts per split, the malicious share of each split, and how large the biggest domain is
counts = items.pivot_table(index="split", columns="cls", values="file", aggfunc="count").loc[SPLIT_NAMES]
counts["malicious share of split"] = counts["malicious"] / (counts["benign"] + counts["malicious"])
counts["benign share of class"] = counts["benign"] / counts["benign"].sum()
counts["malicious share of class"] = counts["malicious"] / counts["malicious"].sum()
counts["largest domain, share of split rows"] = pd.Series({s: g["domain"].value_counts(normalize=True).iloc[0] for s, g in items.groupby("split")})
counts.round(4)

cls,benign,malicious,malicious share of split,benign share of class,malicious share of class,"largest domain, share of split rows"
split,,,,,,
train,300969,403020,0.5725,0.70,0.70,0.021
validation,64494,86361,0.5725,0.15,0.15,0.000
test,64494,86362,0.5725,0.15,0.15,0.000


**Reading.** Each class lands close to 70/15/15, and every split is about 57% malicious, so the AUPRC of a random scorer is about 0.57 in all three. No domain is split across train, validation and test, and no validation or test row shares a host with a training row of its own class. The largest domain (about 2% of the train rows) sits in train, and the largest domain in validation or test is a negligible share, as intended.

Two things to know about the grouping:

- A host that is an IP address has no registered domain, so the `domain` column holds the last two numbers of the address (for example `115.240`). Rows of the same IP always stay together, and unrelated IPs that share those two numbers move together too.
- The classes are split separately, so a domain that appears in both classes can land in different splits. The next cell counts how often this affects malicious validation and test rows.

In [13]:
# Malicious validation and test rows whose domain also appears in the benign train split
benign_train_domains = set(items.loc[(items["cls"] == "benign") & (items["split"] == "train"), "domain"])
mal_eval = items[(items["cls"] == "malicious") & (items["split"] != "train")]
shared = mal_eval["domain"].isin(benign_train_domains)
print(f"{shared.sum():,} of {len(mal_eval):,} malicious validation and test rows ({shared.mean():.3%}) share a domain with a benign train row")

99 of 172,723 malicious validation and test rows (0.057%) share a domain with a benign train row


## 6. Codes outside the canvas, per split

The image models use only the rows with `fits_canvas` true, and the excluded codes are reported per split. URL models use every row.

In [14]:
# Codes outside the 65 x 65 canvas, per split and class
excluded_split = items.groupby(["split", "cls"])["fits_canvas"].agg(rows="size", outside_canvas=lambda s: int((~s).sum()))
excluded_split["outside share"] = excluded_split["outside_canvas"] / excluded_split["rows"]
excluded_split.loc[SPLIT_NAMES].round(4)

rows  outside_canvas  outside share
split      cls                                             
train      benign     300969             106         0.0004
           malicious  403020            4271         0.0106
validation benign      64494              23         0.0004
           malicious   86361             113         0.0013
test       benign      64494              28         0.0004
           malicious   86362             122         0.0014

## 7. What uses which rows

| Model | Fitted on | Tuned on | Threshold set on | Scored on |
| --- | --- | --- | --- | --- |
| Floors and the VAE | benign train rows | validation rows | benign validation rows | validation and test rows |
| Supervised reference | benign and malicious train rows | validation rows | benign validation rows | validation and test rows |

The test rows are used once per model, after the configuration is fixed on validation. Malicious train rows are used only by the supervised reference.

The model notebooks read the items table with `load_items` and take these rows from it. The image models pass `fits_canvas_only=True`, and the URL models use every row.

### Files this notebook saved in `data/processed/`

| File | What it holds |
| --- | --- |
| `image_grid_sizes.csv.gz` | grid size and canvas fit of every kept image |
| `canvas_exclusion_summary.json` | counts of excluded codes per class |
| `image_features.csv.gz` | modules, quiet zone and dark fraction of every kept image |
| `items.csv.gz` | the items table read by the model notebooks |
| `items_summary.json` | counts per split, excluded and truncated codes, and a short `data_version` that changes if the rows or splits change |